Ten skrypt ładuje dane do schematu hurtowni danych

In [0]:
%sql
MERGE INTO dw.dim_car AS Dim_car
USING (
  SELECT
    Inventory.inventory_id,
    CASE WHEN Car.last_update > Inventory.last_update THEN Car.last_update ELSE Inventory.last_update END AS new_valid_to
  FROM landing.car AS Car
  INNER JOIN landing.inventory AS Inventory ON Car.car_id = Inventory.car_id
) AS src
ON Dim_car.car_inventory_id = src.inventory_id
AND Dim_car.is_active = true
AND (Dim_car.valid_from < src.new_valid_to)
WHEN MATCHED THEN
  UPDATE SET
    is_active = false,
    valid_to = src.new_valid_to,
    dw_update_date = CURRENT_TIMESTAMP()

In [0]:
%sql
INSERT INTO dw.dim_car (car_inventory_id, car_id, car_producer, car_model, car_production_year, car_fuel_type, car_license_plates, car_current_rental_rate, car_purchase_price, car_sell_price, valid_from, valid_to, is_active, dw_insert_date, dw_update_date)
SELECT
    Inventory.inventory_id AS car_inventory_id,
    Car.car_id,
    Car.producer,
    Car.model,
    Inventory.production_year,
    Inventory.fuel_type,
    Inventory.license_plates,
    Car.rental_rate AS current_rental_rate,
    Inventory.purchase_price,
    Inventory.sell_price,
    CASE WHEN Car.last_update > Inventory.last_update THEN Car.last_update
        ELSE Inventory.last_update END AS valid_from,
    '9999-12-31' AS valid_to,
    true AS is_active,
    CURRENT_TIMESTAMP() AS dw_insert_date,
    NULL AS dw_update_date
FROM landing.car Car
INNER JOIN landing.inventory Inventory ON Car.car_id = Inventory.car_id
LEFT JOIN dw.dim_car Dim_car ON Dim_car.car_inventory_id = Inventory.inventory_id AND Dim_car.is_active = true
WHERE Dim_car.car_inventory_id IS NULL;


In [0]:
%sql
INSERT INTO dw.fact_service (service_id, car_key, service_date_key, service_type, service_cost)
SELECT 
    Service.service_id,
    Dim_car.car_key,
    Dim_service_date.service_date_key,
    Service.service_type,
    Service.service_cost
FROM landing.service Service
INNER JOIN dw.dim_car Dim_car ON Dim_car.car_inventory_id = Service.inventory_id
INNER JOIN dw.dim_service_date Dim_service_date ON Dim_service_date.service_date = Service.service_date
LEFT JOIN dw.fact_service Fact_service ON Fact_service.service_id = Service.service_id
WHERE Fact_service.service_id IS NULL;